# Pytorch 익히기

In [1]:
import torch

Tensor 생성

In [2]:
temp = torch.tensor([[1, 2], [3, 4]])

In [3]:
temp

tensor([[1, 2],
        [3, 4]])

In [4]:
temp.shape, temp.view((4,1))

(torch.Size([2, 2]),
 tensor([[1],
         [2],
         [3],
         [4]]))

1차원 vs 2차원 return

In [5]:
temp.view(-1), temp.view(1,-1), temp.view(-1,1)

(tensor([1, 2, 3, 4]),
 tensor([[1, 2, 3, 4]]),
 tensor([[1],
         [2],
         [3],
         [4]]))

# Custom Dataset
● 딥러닝은 기본적으로 대량의 데이터를 이용하여 모델을 학습

● 데이터를 한 번에 메모리에 불러와서 훈련시키면 시간과 비용 측면에서 효율적이지
않음

● 데이터를 한 번에 다 부르지 않고 조금씩 나누어 불러서 사용하는 방식이 custom dataset

In [6]:
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [7]:
class CustomDataset(Dataset):

  def __init__(self):
    # 필요한 변수를 선언하고 데이터셋을 전처리하는 부분 (예시 데이터: y = 2x + 1)
    self.x_data = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0]], dtype=np.float32)
    self.y_data = np.array([[3.0], [5.0], [7.0], [9.0], [11.0], [13.0]], dtype=np.float32)

  def __len__(self):
    return len(self.x_data)

  def __getitem__(self, index):
    # 데이터셋의 길이(총 샘플 수) 반환
    # index번째 데이터를 텐서 형태로 반환
    x = torch.FloatTensor(self.x_data[index])
    y = torch.FloatTensor(self.y_data[index])
    return x, y

dataset = CustomDataset()
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

for i, (x, y) in enumerate(dataloader):
  print(f"batch {i}: x={x.view(-1).tolist()}, y={y.view(-1).tolist()}")

batch 0: x=[1.0, 4.0], y=[3.0, 9.0]
batch 1: x=[6.0, 5.0], y=[13.0, 11.0]
batch 2: x=[3.0, 2.0], y=[7.0, 5.0]


# MNIST 데이터셋 내려받기

In [8]:
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
import requests


In [9]:
mnist_transform = transforms.Compose([
transforms.ToTensor(), # 이미지를 텐서로 변환
transforms.Normalize((0.5,), (1.0,)) # 평균 0.5, 표준편차 1.0으로 정규화
])

download_root = './data/MNIST_DATASET' # 내려받을 경로 지정
train_dataset = MNIST(download_root, transform=mnist_transform, train=True, download=True)
valid_dataset = MNIST(download_root, transform=mnist_transform, train=False, download=True)
test_dataset = MNIST(download_root, transform=mnist_transform, train=False, download=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 484kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 14.4MB/s]


## 데이터셋 확인

In [10]:
print(train_dataset)

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data/MNIST_DATASET
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5,), std=(1.0,))
           )


In [11]:
print(len(train_dataset), len(valid_dataset), len(test_dataset))

60000 10000 10000


# 모델 정의 (Multi Layer Perceptron)

In [12]:
import torch.nn as nn

In [13]:
class MLP(nn.Module):


  def __init__(self):

    super(MLP, self).__init__()

    self.layer1 = nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=64, kernel_size=5),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2)
    )

    self.layer2 = nn.Sequential(
        nn.Conv2d(in_channels=64, out_channels=30, kernel_size=5),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2)
    )

    self.layer3 = nn.Sequential(
        nn.Linear(in_features=30*5*5, out_features=10, bias=True),
        nn.ReLU(inplace=True)
    )


  def forward(self, x):
    x = self.layer1(x)
    x = self.layer2(x)
    x = x.view(x.shape[0], -1)
    x = self.layer3(x)
    return x


In [14]:
model = MLP()

In [15]:
print("children")
list(model.children())

children


[Sequential(
   (0): Conv2d(3, 64, kernel_size=(5, 5), stride=(1, 1))
   (1): ReLU(inplace=True)
   (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 ),
 Sequential(
   (0): Conv2d(64, 30, kernel_size=(5, 5), stride=(1, 1))
   (1): ReLU(inplace=True)
   (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 ),
 Sequential(
   (0): Linear(in_features=750, out_features=10, bias=True)
   (1): ReLU(inplace=True)
 )]

## Parameter : weight 등
## Hyper Parameter : 사람이 직접 정의하는거 (batch size 등)
## Optimizer : parameter를 업데이트 하는전략

# 모델 Parameter 정의

In [16]:
model = nn.Linear(in_features=1, out_features=1, bias=True) # 학습에 사용할 단순 선형 모델

criterion = torch.nn.MSELoss() # 손실 함수 (회귀이므로 MSELoss)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda epoch: 0.95 ** epoch)

loss_fn = criterion

for epoch in range(1, 100 + 1): # 에포크 수만큼 데이터를 반복하여 처리
  for x, y in dataloader: # 배치 크기만큼 데이터를 가져와서 학습 진행
    optimizer.zero_grad() # 기울기를 0으로 초기화
    loss_fn(model(x), y).backward() # 오차 역전파로 기울기 자동 계산
    optimizer.step() # 파라미터 업데이트
  scheduler.step() # 에포크마다 학습률 조정

print("학습된 파라미터: w =", model.weight.item(), ", b =", model.bias.item())

학습된 파라미터: w = 2.030081033706665 , b = 0.8710814118385315


# 모델 학습

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = nn.Sequential(
    nn.Flatten(), # (N, 1, 28, 28) → (N, 784)   N=batch_size    1=channel수(3이면 rgb)
    nn.Linear(28 * 28, 256), nn.ReLU(),
    nn.Linear(256, 10) # 10개 클래스 로짓 출력    0~9 10개
).to(device)      # 지금 cuda / cpu 사용

criterion = torch.nn.CrossEntropyLoss() # 다중 클래스 분류 손실 함수

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

num_epochs = 10 # Define the number of epochs

for epoch in range(1, num_epochs + 1):
  model.train() # 훈련 모드 선언
  for x, y in train_dataloader: # 배치 단위로 학습
    x, y = x.to(device), y.to(device)
    outputs = model(x) # 모델의 예측 값 (로짓)
    loss = criterion(outputs, y) # 예측과 정답 사이의 오차 계산
    optimizer.zero_grad() # 기울기를 0으로 초기화
    loss.backward() # 기울기 자동 계산 (역전파)
    optimizer.step() # 파라미터 업데이트

# 훈련 과정 모니터링

In [18]:
valid_dataloader = DataLoader(valid_dataset, batch_size=64, shuffle=False)

model.eval() # 검증 모드로 전환 (드롭아웃 비활성화)

valid_loss, correct, total = 0.0, 0, 0

with torch.no_grad(): # 기울기를 저장하지 않아 메모리와 연산 시간 절약
  for x, y in valid_dataloader:
    x, y = x.to(device), y.to(device)
    outputs = model(x) # 모델의 예측 값 (로짓)
    loss = criterion(outputs, y) # 검증 손실 계산
    valid_loss += loss.item() * x.size(0)
    correct += (outputs.argmax(dim=1) == y).sum().item() # 예측과 정답 비교
    total += y.size(0)

valid_loss = valid_loss / total # 평균 손실
valid_acc = correct / total

print(f"검증 손실 = {valid_loss:.4f} | 검증 정확도 = {valid_acc:.4f} ({correct}/{total})")

검증 손실 = 0.0737 | 검증 정확도 = 0.9767 (9767/10000)


 ## model.eval()에서 with torch.no_grad()를 사용하는 이유

● 파이토치는 모든 연산과 기울기 값을 저장

● test / validation 과정에서는 역전파가 필요하지 않으므로 with torch.no_grad()를
사용하여 기울기 값을 저장하지 않도록 함

● 이와 같은 과정을 통해 기울기 값을 저장하고 기록하는 데 필요한 메모리와 연산 시간을
줄일 수 있음